# 6. Changing a primitive — and an approximate adder

Every DRAM command sequence in this tutorial comes from **lime**, which compiles a
small *majority network* per primitive (full adder, AND, …) into the shortest AAP/AP
sequence it can find. Those networks are built into the compiler. The option
`--pud-primitives=<file>` replaces any of them with one written in MLIR, so you can
change what a primitive is — and what it costs — without rebuilding anything.

The mapping (which prices a primitive) and the code generator (which emits it) read
the same file, so the cost model and the program cannot disagree.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # tutorial/, for _infra
from _infra import pud

import shutil
ART = pud.ARTIFACTS
ART.mkdir(parents=True, exist_ok=True)
full_adder = ART / "full_adder.mlir"
shutil.copy(pud.ROOT / "cinnamon" / "samples" / "pud-primitives" / "full_adder.mlir", full_adder)
print(full_adder.read_text())

That is the full adder exactly as built in, as a majority network: `bits.maj` is a
three-input majority and `bits.not` a complement. `pud.resident = "DCC1"` keeps the
carry in a dual-contact cell from one bit to the next, and `pud.commands = 8` makes the
compiler refuse the file if the sequence ever takes a different number of commands.
Compiled from the file it gives the same eight commands; only the *source* column
changes:

In [ ]:
print(pud.primitive_table(names=("row_add",)).split("\n\n")[0])
print(pud.primitive_table(flags=(f"--pud-primitives={full_adder}",), names=("row_add",)))

A file cannot redefine what a primitive *means*. Every definition is run on a model of
the substrate against the primitive's function, and refused if it does not compute it.
Here is a "full adder" whose sum is the majority — wrong:

In [ ]:
wrong = ART / "wrong_adder.mlir"
wrong.write_text(full_adder.read_text().replace(
    "return %sum, %cout", "return %cout, %cout").replace("\n    attributes {pud.commands = 8}", ""))
try:
    pud.primitive_table(flags=(f"--pud-primitives={wrong}",))
except RuntimeError as e:
    print(next(l for l in str(e).splitlines() if "LLVM ERROR" in l))

## An approximate adder

Exact addition is a carry chain: 8 commands per bit on this substrate, because every bit
waits for the carry of the one below. Many applications tolerate small errors in the low
bits. A classic approximation, the **lower-part OR adder**, computes the low *K* bits
without any carry — each result bit is `a | b` — and adds the upper bits exactly, starting
from carry 0:

| bits | computed by | commands per bit |
|---|---|---|
| 0 … K−1 | the approximate cell, `row_add_approx` (built in: `a \| b`) | 4 |
| K … n−1 | the full adder, carry starting at 0 | 8 |

`--pud-approx-add-low-bits=K` does that for every plain addition in a kernel
(reductions and multiplications stay exact). For a 16-bit add, a tile then costs
`4K + 8(16 − K) + 1` commands instead of 129.

The cell is a primitive like any other, so its network can come from a file too. To
make the accuracy numbers below come from exactly what the compiler compiles, we write
the built-in OR cell as a file and use it throughout:

In [ ]:
R = "!bits.row<1>"
def cell_file(name, body, result):
    path = ART / f"cell_{name}.mlir"
    path.write_text(
        f"func.func @row_add_approx(%a: {R}, %b: {R}) -> {R} {{\n"
        f"  %w = arith.constant 1 : i64\n"
        f"  %zero = bits.create_all_zero_row %w : i64 -> {R}\n"
        f"  %one = bits.create_all_one_row %w : i64 -> {R}\n"
        f"{body}"
        f"  return {result} : {R}\n}}\n")
    return path

M = f"{R}, {R}, {R} -> {R}"
cells = {
    "or":   cell_file("or", f"  %s = bits.maj %a, %b, %one : {M}\n", "%s"),
    "xor":  cell_file("xor", f"  %o = bits.maj %a, %b, %one : {M}\n"
                             f"  %n = bits.maj %a, %b, %zero : {M}\n"
                             f"  %nn = bits.not %n : {R} -> {R}\n"
                             f"  %s = bits.maj %o, %nn, %zero : {M}\n", "%s"),
    "copy": cell_file("copy", "", "%a"),
}
print(cells["or"].read_text())
print(pud.primitive_table(flags=(f"--pud-primitives={cells['or']}",), names=("row_add_approx",)))

## Cost and accuracy as *K* grows

For each *K*, `vadd` is compiled at 32 banks and run on gem5. Its accuracy comes from
evaluating the same cell file in Python over 200,000 random pairs of 15-bit operands
(so the exact sum still fits in 16 bits), because reading a compiled kernel's results
back is not possible yet (see the end of this notebook).

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
a = rng.integers(0, 1 << 15, 200_000, dtype=np.int64)
b = rng.integers(0, 1 << 15, 200_000, dtype=np.int64)
exact = a + b

def accuracy(low_bits, cell_path):
    cell = pud.network_function(cell_path, "row_add_approx")
    got = pud.approx_add(a, b, 16, low_bits, cell)
    err = np.abs(got - exact)
    return dict(error_rate=float(np.mean(err != 0)), mean_abs=float(err.mean()),
                max_abs=int(err.max()), mred=float(np.mean(err / np.maximum(exact, 1))))

def run(low_bits, cell="or", banks=32):
    flags = (f"--pud-primitives={cells[cell]}", f"--pud-approx-add-low-bits={low_bits}")
    k = pud.compile_kernel("vadd_1048576", banks, flags=flags, tag=f"vadd_approx_{cell}_K{low_bits}")
    return k, pud.simulate(k.exe)

rows = []
for K in (0, 4, 8, 12):
    k, s = run(K)
    rows.append(dict(K=K, row_ops=s.row_ops, expected=128 * (4 * K + 8 * (16 - K) + 1),
                     us=s.makespan_us, **accuracy(K, cells["or"])))
    print(f"K={K:2} done")

In [ ]:
base = rows[0]["us"]
print(f"{'K':>3} {'row ops':>8} {'expected':>8} {'makespan us':>11} {'speed-up':>8} "
      f"{'wrong':>7} {'mean |err|':>10} {'max |err|':>9} {'MRED':>8}")
for r in rows:
    print(f"{r['K']:3} {r['row_ops']:8} {r['expected']:8} {r['us']:11.3f} {base / r['us']:8.2f} "
          f"{r['error_rate']:7.1%} {r['mean_abs']:10.2f} {r['max_abs']:9} {r['mred']:8.2e}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot([r["mred"] for r in rows[1:]], [base / r["us"] for r in rows[1:]], "o-")
for r in rows[1:]:
    ax.annotate(f"K={r['K']}", (r["mred"], base / r["us"]), textcoords="offset points", xytext=(5, -12))
ax.set_xscale("log"); ax.set_xlabel("mean relative error distance"); ax.set_ylabel("speed-up over exact")
ax.set_title("vadd, lower-part OR adder, 32 banks"); plt.tight_layout()

**What to look for.**

- The command count gem5 measures is exactly `128 × (4K + 8(16 − K) + 1)`: the mapping
  priced the approximate cell, and the program issued it.
- Each approximate bit saves four commands, so the makespan falls roughly linearly in *K*.
- The error grows by about a factor of 16 for every four more approximate bits: an error
  in bit *K − 1* is worth 2^(K−1).

## Which cell?

The OR cell is only one choice. With the file you can try others — here at *K* = 8:
`a ^ b`, the sum bit of a half adder, and simply copying `a`, ignoring `b` in the low
bits.

In [ ]:
cmp = []
for name in ("or", "xor", "copy"):
    table = pud.primitive_table(flags=(f"--pud-primitives={cells[name]}",), names=("row_add_approx",))
    per_bit = int(table.splitlines()[1].split()[1])
    k, s = run(8, name)
    cmp.append(dict(cell=name, per_bit=per_bit, row_ops=s.row_ops, us=s.makespan_us, **accuracy(8, cells[name])))
print(f"{'cell':>5} {'cmds/bit':>8} {'row ops':>8} {'makespan us':>11} {'speed-up':>8} {'wrong':>7} {'MRED':>8}")
for r in cmp:
    print(f"{r['cell']:>5} {r['per_bit']:8} {r['row_ops']:8} {r['us']:11.3f} {base / r['us']:8.2f} "
          f"{r['error_rate']:7.1%} {r['mred']:8.2e}")

**XOR is worse on both counts.** It takes as many commands as the full adder, so it
saves nothing, and its error is twice OR's. Where both bits are 1 the true two-bit sum is
`10`: OR answers `1`, off by one, while XOR answers `0`, off by two. Where the bits differ
both are right.

**The copy is the cheapest cell,** one command a bit, and the fastest; its error is about
XOR's. So among these three, OR is the most accurate for its cost, and the copy is what to
reach for when speed matters more — the kind of trade-off a primitive file lets you explore
without touching the compiler. An approximation only pays if its cell is cheaper than the
full adder it replaces.

> **Try it:** write your own cell with `cell_file("mine", ...)` and compare it. Any
> network of `bits.maj` / `bits.not` over `%a`, `%b`, `%zero` and `%one` works; the
> compiler proves the sequence it emits computes your network.

## What this does and does not show

The command counts and times are measured: gem5 ran the compiled programs. The command
sequence for each cell is proved by the compiler to compute that cell's network, and gem5
now carries out what the commands do to the rows. The **accuracy**, though, is computed in
Python from the same network, not read back from gem5: loading a kernel's data into the
bit-sliced layout and reading results out is the open step described in `HANDOVER.md`.